# Chemistry PDF extractor (Kaggle)

**Code**: public GitHub (cloned below). **PDFs**: Kaggle dataset only.

Notebook settings: **GPU** + **Internet**.

PDFs may live in subfolders inside the dataset zip — the notebook searches recursively.

If `git clone` fails with *Unable to read current working directory*, use **Restart session** and run cells from the top (a prior run may have deleted the folder you were still inside).

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils git

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

GITHUB_REPO = "https://github.com/sgcharan/Test-Paper-Extractor.git"
GITHUB_BRANCH = "main"
PDF_INPUT_DIR = Path("/kaggle/input/datasets/sgcharan7/cbse-12-checm-samples")

WORKING_DIR = Path("/kaggle/working")
PROJECT_ROOT = WORKING_DIR / "test-paper-extractor"
OUT_ROOT = WORKING_DIR / "out"
DPI = 300
QWEN_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
MAX_VLM_SIDE = 1280

WORKING_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING_DIR)

print("Datasets mounted:", [p.name for p in Path("/kaggle/input").iterdir()])
print("cwd:", os.getcwd())

In [ ]:
import shutil
import subprocess

# Must chdir out of PROJECT_ROOT before deleting it (re-runs otherwise break git)
os.chdir(WORKING_DIR)

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

subprocess.check_call(
    [
        "git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
        GITHUB_REPO, str(PROJECT_ROOT),
    ],
    cwd=str(WORKING_DIR),
)

if not (PROJECT_ROOT / "extract_paper.py").is_file():
    raise FileNotFoundError(f"Repo cloned but extract_paper.py missing under {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pdf_discovery import discover_pdfs, format_tree_hint

pdfs = discover_pdfs(PDF_INPUT_DIR, recursive=True)
if not pdfs:
    raise FileNotFoundError(
        f"No PDFs under {PDF_INPUT_DIR}.\n"
        f"Check PDF_INPUT_DIR matches Add Data slug. Files seen:\n  {format_tree_hint(PDF_INPUT_DIR)}"
    )

print(f"Cloned {GITHUB_REPO}")
print(f"Found {len(pdfs)} PDF(s):")
for p in pdfs:
    print(" ", p.relative_to(PDF_INPUT_DIR))

!pip install -q -r requirements.txt

In [ ]:
from batch_extract import main as batch_main

OUT_ROOT.mkdir(parents=True, exist_ok=True)
argv = [
    str(PDF_INPUT_DIR),
    "--out-root", str(OUT_ROOT),
    "--recursive",
    "--backend", "qwen_hf",
    "--dpi", str(DPI),
    "--qwen-model", QWEN_MODEL,
    "--max-vlm-side", str(MAX_VLM_SIDE),
    "--max-new-tokens", "2048",
]
batch_main(argv)

In [ ]:
from preview import build_preview_html

for questions_path in OUT_ROOT.glob("*/questions.json"):
    print("Preview:", build_preview_html(questions_path))

In [ ]:
import shutil

zip_path = WORKING_DIR / "out.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUT_ROOT)
print("Download from Output:", zip_path)